<a href="https://colab.research.google.com/github/ldongheedev/-BDA-LLM-RAG-Program/blob/main/4%EC%A3%BC%EC%B0%A8_%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 9과 도전과제 | 영화 리뷰 감성 분석 — 극한직업

> 4주차에 배운 텍스트 분류 파이프라인을 영화 리뷰 데이터에 적용합니다.

```
영화 리뷰 텍스트 → 전처리 → TF-IDF → 머신러닝 모델 → 긍정 😊 / 부정 😞
```
---

In [1]:
!pip install scikit-learn --quiet

import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("✅ 준비 완료!")

✅ 준비 완료!


In [2]:
# 극한직업 긍정 리뷰 25개
긍정_리뷰 = [
    "류승룡의 연기가 너무 웃겨서 배꼽 빠질 뻔했어요 최고의 코미디",
    "치킨집 수사라는 설정이 기발하고 유머가 넘쳐요 강력 추천",
    "온 가족이 함께 보기 딱 좋은 영화예요 내내 웃었어요",
    "코미디인데 형사물 긴장감도 살아있어서 두 배로 재밌었어요",
    "수원 왕갈비 치킨 덕분에 영화 보는 내내 배고팠어요 너무 웃겨요",
    "배우들의 케미가 폭발해요 팀워크가 느껴져서 더 재밌었어요",
    "결말이 통쾌하고 유쾌해서 극장 나올 때 기분이 너무 좋았어요",
    "대사 하나하나가 다 명대사예요 두고두고 생각나는 영화",
    "이런 발상을 어떻게 했지 아이디어가 천재적인 영화예요",
    "웃음 포인트가 쉴 새 없이 나와서 지루할 틈이 없었어요",
    "류승룡이 치킨 굽는 장면이 진짜 명장면이에요 너무 웃겼어요",
    "이동휘의 애드리브가 빛났어요 영화 내내 포복절도했습니다",
    "한국 코미디 영화 역대급이라고 생각해요 진짜 걸작",
    "천만 관객 기록이 납득되는 영화예요 모두가 좋아할 수밖에 없어요",
    "스토리가 탄탄하면서도 개그가 찰져서 완성도가 높아요",
    "마약 범죄라는 진지한 소재를 이렇게 코믹하게 풀다니 놀라워요",
    "중간에 나오는 갈비 냄새 맡는 장면에서 진짜 소리 내서 웃었어요",
    "극 중 팀원들이 각자 개성이 뚜렷해서 캐릭터가 살아있어요",
    "오락 영화의 정수를 보여주는 영화 지루한 순간이 단 한 순간도 없어요",
    "두 번 봐도 새로운 웃음 포인트가 나오는 영화예요 재관람 강추",
    "류승룡과 이하늬의 부부 케미가 찰떡이에요 너무 웃기고 귀여워요",
    "한국형 코미디의 새로운 기준을 제시한 영화라고 생각해요",
    "시나리오가 정말 잘 짜여 있어요 복선도 있고 반전도 있고 완벽해요",
    "우리나라 형사들이 다 이렇게 열정적이면 좋겠다 하는 생각이 들었어요",
    "보고 나서 치킨이 먹고 싶어지는 부작용이 있는 영화 최고예요",
]

# 극한직업 부정 리뷰 25개
부정_리뷰 = [
    "개그가 너무 유치하고 과장이 심해서 공감이 안 됐어요",
    "스토리가 말이 안 되고 개연성이 너무 부족했어요",
    "코미디라는 핑계로 허술한 스토리를 덮으려는 느낌이 강했어요",
    "천만이 봤다고 해서 기대했는데 제 취향이 아니었어요 실망이에요",
    "억지 웃음 포인트가 많아서 오히려 민망했어요",
    "마약 수사 내용이 너무 가볍게 다뤄진 것 같아서 불편했어요",
    "류승룡 원맨쇼 느낌이 강해서 나머지 배우들이 너무 아깝게 쓰였어요",
    "중반부 이후 개그가 반복되는 느낌이라 지루해졌어요",
    "비슷한 유형의 코미디를 너무 많이 봐서 신선함이 없었어요",
    "결말이 너무 허무하고 급하게 마무리된 느낌이에요",
    "억지스러운 상황 설정이 몰입을 방해했어요",
    "웃기긴 한데 딱 거기서 끝나요 감동이나 여운이 전혀 없어요",
    "과대평가된 영화라고 생각해요 코미디 치고 웃기지도 않았어요",
    "캐릭터들이 너무 전형적이고 식상해서 신선함이 없었어요",
    "형사물로서의 긴장감이 거의 없어서 장르의 매력을 못 살렸어요",
    "유머 코드가 너무 올드해서 젊은 사람들 취향이 아닌 것 같아요",
    "치킨집 설정 하나로 너무 우려먹은 느낌이에요",
    "빌런 캐릭터가 너무 약해서 긴장감이 전혀 없었어요",
    "기대가 너무 컸나 주변에서 다들 재밌다 해서 봤는데 별로였어요",
    "코미디 장면이 너무 길어서 중간에 지루함을 느꼈어요",
    "여성 캐릭터가 너무 전형적으로 그려진 것 같아서 아쉬웠어요",
    "전반적으로 뭔가 어설픈 느낌이 계속 들었어요",
    "개봉 당시 분위기에 편승해서 과대평가된 영화 같아요",
    "두 시간이 생각보다 길게 느껴졌어요 편집이 아쉬웠어요",
    "설정은 참신한데 그걸 살리지 못한 아쉬운 영화예요",
]

X = 긍정_리뷰 + 부정_리뷰
y = [1]*25 + [0]*25

print(f"전체 데이터: {len(X)}개  (긍정 {sum(y)}개 / 부정 {len(y)-sum(y)}개)")

전체 데이터: 50개  (긍정 25개 / 부정 25개)


In [3]:
# 전처리
def 전처리(텍스트):
    텍스트 = re.sub(r'[^가-힣a-zA-Z0-9 ]', ' ', 텍스트)
    return re.sub(r'\s+', ' ', 텍스트).strip()

X = [전처리(r) for r in X]

print("전처리 완료!")
print("예시:", X[0])

전처리 완료!
예시: 류승룡의 연기가 너무 웃겨서 배꼽 빠질 뻔했어요 최고의 코미디


In [4]:
# 훈련 / 테스트 분리 (75% / 25%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f"훈련 데이터: {len(X_train)}개  /  테스트 데이터: {len(X_test)}개")
print(f"훈련 — 긍정: {sum(y_train)}개, 부정: {len(y_train)-sum(y_train)}개")
print(f"테스트 — 긍정: {sum(y_test)}개, 부정: {len(y_test)-sum(y_test)}개")

훈련 데이터: 37개  /  테스트 데이터: 13개
훈련 — 긍정: 20개, 부정: 17개
테스트 — 긍정: 5개, 부정: 8개


In [5]:
# TF-IDF 변환 (훈련만 fit, 테스트는 transform만)
변환기 = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
X_train_vec = 변환기.fit_transform(X_train)
X_test_vec  = 변환기.transform(X_test)

print(f"훈련 TF-IDF 행렬: {X_train_vec.shape}")
print(f"  → {X_train_vec.shape[0]}개 리뷰 × {X_train_vec.shape[1]}개 특성")

훈련 TF-IDF 행렬: (37, 1777)
  → 37개 리뷰 × 1777개 특성


In [6]:
# 모델 학습 (나이브 베이즈 & 로지스틱 회귀)
모델_NB = MultinomialNB()
모델_NB.fit(X_train_vec, y_train)
예측_NB = 모델_NB.predict(X_test_vec)

모델_LR = LogisticRegression(max_iter=1000, random_state=42)
모델_LR.fit(X_train_vec, y_train)
예측_LR = 모델_LR.predict(X_test_vec)

print("학습 완료!")

학습 완료!


In [7]:
# 정확도 비교
nb_acc = accuracy_score(y_test, 예측_NB)
lr_acc = accuracy_score(y_test, 예측_LR)

print(f"나이브 베이즈:  {nb_acc:.2%}  {'█'*int(nb_acc*20)}")
print(f"로지스틱 회귀: {lr_acc:.2%}  {'█'*int(lr_acc*20)}")
print()

if nb_acc >= lr_acc:
    print("→ 이 데이터에서는 나이브 베이즈가 더 좋거나 동일합니다")
else:
    print("→ 이 데이터에서는 로지스틱 회귀가 더 좋습니다")

나이브 베이즈:  53.85%  ██████████
로지스틱 회귀: 46.15%  █████████

→ 이 데이터에서는 나이브 베이즈가 더 좋거나 동일합니다


In [8]:
# 상세 성능 (Precision / Recall / F1)
print("=" * 42)
print("나이브 베이즈 상세 성능")
print("=" * 42)
print(classification_report(y_test, 예측_NB,
                              target_names=["부정", "긍정"], zero_division=0))

print("=" * 42)
print("로지스틱 회귀 상세 성능")
print("=" * 42)
print(classification_report(y_test, 예측_LR,
                              target_names=["부정", "긍정"], zero_division=0))

나이브 베이즈 상세 성능
              precision    recall  f1-score   support

          부정       0.75      0.38      0.50         8
          긍정       0.44      0.80      0.57         5

    accuracy                           0.54        13
   macro avg       0.60      0.59      0.54        13
weighted avg       0.63      0.54      0.53        13

로지스틱 회귀 상세 성능
              precision    recall  f1-score   support

          부정       1.00      0.12      0.22         8
          긍정       0.42      1.00      0.59         5

    accuracy                           0.46        13
   macro avg       0.71      0.56      0.41        13
weighted avg       0.78      0.46      0.36        13



In [9]:
# 혼동 행렬
cm = confusion_matrix(y_test, 예측_LR)

print("혼동 행렬 (로지스틱 회귀):")
print()
print("              예측:부정   예측:긍정")
print(f"  실제:부정      {cm[0][0]:3}        {cm[0][1]:3}")
print(f"  실제:긍정      {cm[1][0]:3}        {cm[1][1]:3}")
print()
print(f"  TN(✅ 부정→부정): {cm[0][0]}개")
print(f"  FP(❌ 부정→긍정): {cm[0][1]}개  ← 부정 리뷰를 긍정으로 착각")
print(f"  FN(❌ 긍정→부정): {cm[1][0]}개  ← 긍정 리뷰를 부정으로 착각")
print(f"  TP(✅ 긍정→긍정): {cm[1][1]}개")

혼동 행렬 (로지스틱 회귀):

              예측:부정   예측:긍정
  실제:부정        1          7
  실제:긍정        0          5

  TN(✅ 부정→부정): 1개
  FP(❌ 부정→긍정): 7개  ← 부정 리뷰를 긍정으로 착각
  FN(❌ 긍정→부정): 0개  ← 긍정 리뷰를 부정으로 착각
  TP(✅ 긍정→긍정): 5개


In [10]:
# 가중치 분석 — 모델이 판단에 사용한 핵심 단어 패턴
#
# 전략:
#   1) 조사·어미만으로 이루어진 의미 없는 패턴 제거
#   2) 단어 경계(공백 포함) n-gram으로 읽기 쉬운 단어 단위 패턴 우선 추출
#   3) 글자 단위 패턴도 별도로 확인하여 더 다양한 단어 패턴을 보여줌

어미_조사 = {
    '어요','해요','하고','게요','이고','이다','이에','있고','어있','고요',
    '이요','네요','에요','라고','으로','에서','지만','는데','으면','아요',
    '겠어','하여','하며','이며','이라','이나','라서','아서','어서','해서',
    '하면','이면','라면','에게','으며','으나','이야','이죠','했어','됐어',
    '갔어','왔어','있어','없어','같아','싶어','봐요','줘요','하죠','이죠',
    '라죠','하네','이네','라네','다네','했네','됐네','같이','처럼','보다',
    '부터','까지','마다','조차','라도','이나','만큼','뿐이','뿐만',
}

def 의미있는_패턴(p):
    p_clean = p.strip()
    if len(p_clean) <= 1:
        return False
    if p_clean in 어미_조사:
        return False
    if p.endswith(' ') and p.strip() in 어미_조사:
        return False
    # 공백으로만 이루어진 패턴 제거
    if p_clean == '':
        return False
    return True

feature_names = 변환기.get_feature_names_out()
가중치 = 모델_LR.coef_[0]

# 의미 있는 패턴 전체
의미_idx      = [i for i, p in enumerate(feature_names) if 의미있는_패턴(p)]
의미_feature  = feature_names[의미_idx]
의미_가중치   = 가중치[의미_idx]

# 단어 경계 포함 패턴 (공백이 포함된 n-gram → 단어 단위로 읽히는 패턴)
단어경계_idx      = [i for i, p in enumerate(의미_feature) if ' ' in p.strip()]
단어경계_feature  = 의미_feature[단어경계_idx]
단어경계_가중치   = 의미_가중치[단어경계_idx]

TOP_N = 15

print("━" * 52)
print(" 긍정에 강한 단어 패턴 TOP 15")
print("━" * 52)
for i in 단어경계_가중치.argsort()[-TOP_N:][::-1]:
    막대 = "█" * min(int(단어경계_가중치[i] * 5), 20)
    print(f"  '{단어경계_feature[i].strip()}':  {단어경계_가중치[i]:+.3f}  {막대}")

print()
print("━" * 52)
print(" 부정에 강한 단어 패턴 TOP 15")
print("━" * 52)
for i in 단어경계_가중치.argsort()[:TOP_N]:
    막대 = "█" * min(int(abs(단어경계_가중치[i]) * 5), 20)
    print(f"  '{단어경계_feature[i].strip()}':  {단어경계_가중치[i]:+.3f}  {막대}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 긍정에 강한 단어 패턴 TOP 15
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 부정에 강한 단어 패턴 TOP 15
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [11]:
# 글자 단위 패턴 추가 확인 (더 다양한 핵심 키워드 파악)
print("━" * 52)
print(" 긍정 핵심 글자 패턴 TOP 15")
print("━" * 52)
for i in 의미_가중치.argsort()[-TOP_N:][::-1]:
    막대 = "█" * min(int(의미_가중치[i] * 5), 20)
    print(f"  '{의미_feature[i].strip()}':  {의미_가중치[i]:+.3f}  {막대}")

print()
print("━" * 52)
print(" 부정 핵심 글자 패턴 TOP 15")
print("━" * 52)
for i in 의미_가중치.argsort()[:TOP_N]:
    막대 = "█" * min(int(abs(의미_가중치[i]) * 5), 20)
    print(f"  '{의미_feature[i].strip()}':  {의미_가중치[i]:+.3f}  {막대}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 긍정 핵심 글자 패턴 TOP 15
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  '영화':  +0.137  
  '영화':  +0.137  
  '생각':  +0.108  
  '생각':  +0.108  
  '예요':  +0.087  
  '예요':  +0.087  
  '웃었':  +0.086  
  '웃었어':  +0.086  
  '웃었':  +0.086  
  '웃었어':  +0.086  
  '웃었어요':  +0.086  
  '대사':  +0.086  
  '두고':  +0.086  
  '진짜':  +0.085  
  '진짜':  +0.085  

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 부정 핵심 글자 패턴 TOP 15
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  '느낌이':  -0.215  █
  '느낌':  -0.215  █
  '느낌':  -0.215  █
  '느낌이':  -0.215  █
  '낌이':  -0.215  █
  '너무':  -0.169  
  '너무':  -0.169  
  '너무':  -0.169  
  '너무':  -0.169  
  '느낌이':  -0.141  
  '낌이':  -0.141  
  '억지':  -0.118  
  '억지':  -0.118  
  '느낌이에':  -0.114  
  '낌이에':  -0.114  


In [12]:
# 새 리뷰 예측
def 예측하기(새_리뷰, 모델):
    처리   = 전처리(새_리뷰)
    벡터   = 변환기.transform([처리])     # transform만! (fit 하면 안 됨)
    예측값 = 모델.predict(벡터)[0]
    확률값 = 모델.predict_proba(벡터)[0]
    결과   = "긍정 😊" if 예측값 == 1 else "부정 😞"
    return 결과, 확률값[1]

테스트_리뷰들 = [
    "류승룡 연기가 진짜 최고예요 너무 웃겼어요",
    "스토리가 너무 허술하고 개연성이 없어서 실망했어요",
    "웃기긴 한데 딱 거기서 끝나는 영화예요",
    "치킨집 설정이 천재적이에요 강력 추천합니다",
    "기대만큼은 아니었지만 나쁘진 않았어요",
]

print("새 리뷰 예측 결과:")
print("=" * 60)
for 리뷰 in 테스트_리뷰들:
    결과_nb, 확률_nb = 예측하기(리뷰, 모델_NB)
    결과_lr, 확률_lr = 예측하기(리뷰, 모델_LR)
    print(f"리뷰: '{리뷰}'")
    print(f"  나이브 베이즈:  {결과_nb}  (긍정 {확률_nb:.1%})")
    print(f"  로지스틱 회귀: {결과_lr}  (긍정 {확률_lr:.1%})")
    print()

새 리뷰 예측 결과:
리뷰: '류승룡 연기가 진짜 최고예요 너무 웃겼어요'
  나이브 베이즈:  긍정 😊  (긍정 68.7%)
  로지스틱 회귀: 긍정 😊  (긍정 61.6%)

리뷰: '스토리가 너무 허술하고 개연성이 없어서 실망했어요'
  나이브 베이즈:  부정 😞  (긍정 44.0%)
  로지스틱 회귀: 부정 😞  (긍정 49.9%)

리뷰: '웃기긴 한데 딱 거기서 끝나는 영화예요'
  나이브 베이즈:  부정 😞  (긍정 48.2%)
  로지스틱 회귀: 긍정 😊  (긍정 52.6%)

리뷰: '치킨집 설정이 천재적이에요 강력 추천합니다'
  나이브 베이즈:  긍정 😊  (긍정 56.6%)
  로지스틱 회귀: 긍정 😊  (긍정 56.1%)

리뷰: '기대만큼은 아니었지만 나쁘진 않았어요'
  나이브 베이즈:  부정 😞  (긍정 47.7%)
  로지스틱 회귀: 긍정 😊  (긍정 52.5%)



In [13]:
# ✏️ 직접 입력해서 예측해보세요!
내_리뷰 = "여기에 극한직업에 대한 나만의 리뷰를 입력하세요"

결과, 확률 = 예측하기(내_리뷰, 모델_LR)
print(f"리뷰: '{내_리뷰}'")
print(f"예측: {결과}  (긍정 확률: {확률:.1%})")

리뷰: '여기에 극한직업에 대한 나만의 리뷰를 입력하세요'
예측: 긍정 😊  (긍정 확률: 55.1%)
